In [2]:
import pandas as pd
import pickle, json

selected_prots = ['Q8NHC5',
'P32190',
'Q55AJ4',
'Q6S8J3',
'P0DKI7',
'Q9G6M7',
'P73098',
'Q0E088',
'Q9V3D2',
'Q57LN7',
'Q9V2T8',
'Q02226',
'Q05481',
'Q9C8G9',
'Q8H7U1',
'Q64268',
'P28037',
'P0C520',
'Q8BGX0',
'P36204',
'P21242',
'P55823',
'P59535',
'Q2KNE5',
'Q9ESN2']

In [3]:
#!/usr/bin/env python3

# standard library modules
import sys, errno, re, json, ssl
from urllib import request
from urllib.error import HTTPError
from time import sleep

def parse_items(items):
  if type(items)==list:
    return ",".join(items)
  return ""
def parse_member_databases(dbs):
  if type(dbs)==dict:
    return ";".join([f"{db}:{','.join(dbs[db])}" for db in dbs.keys()])
  return ""
def parse_go_terms(gos):
  if type(gos)==list:
    return ",".join([go["identifier"] for go in gos])
  return ""
def parse_locations(locations):
  if type(locations)==list:
    return ",".join(
      [",".join([f"{fragment['start']}..{fragment['end']}" 
                for fragment in location["fragments"]
                ])
      for location in locations
      ])
  return ""
def parse_group_column(values, selector):
  return ",".join([parse_column(value, selector) for value in values])

def parse_column(value, selector):
  if value is None:
    return ""
  elif "member_databases" in selector:
    return parse_member_databases(value)
  elif "go_terms" in selector: 
    return parse_go_terms(value)
  elif "children" in selector: 
    return parse_items(value)
  elif "locations" in selector:
    return parse_locations(value)
  return str(value)

def output_list(accession_num):
  #disable SSL verification to avoid config issues
    context = ssl._create_unverified_context()

    next = f"https://www.ebi.ac.uk:443/interpro/api/entry/InterPro/protein/reviewed/{accession_num}/?page_size=200"

    last_page = False
    row_l = []
    attempts = 0
    while next:
      try:
          req = request.Request(next, headers={"Accept": "application/json"})
          res = request.urlopen(req, context=context)
          # If the API times out due a long running query
          if res.status == 408:
            # wait just over a minute
            sleep(61)
            # then continue this loop with the same URL
            continue
          elif res.status == 204:
            #no data so leave loop
            break
          payload = json.loads(res.read().decode())
          next = payload["next"]
          attempts = 0
          if not next:
            last_page = True
      except HTTPError as e:
          if e.code == 408:
            sleep(61)
            continue
          else:
            # If there is a different HTTP error, it wil re-try 3 times before failing
            if attempts < 3:
                attempts += 1
                sleep(61)
                continue
            else:
                sys.stderr.write("LAST URL: " + next)
                raise e

      for i, item in enumerate(payload["results"]):
          row = ( (accession_num + "\t")
          + (parse_column(item["metadata"]["name"], 'metadata.name') + "\t")
          + (parse_column(item["metadata"]["source_database"], 'metadata.source_database') + "\t")
          + (parse_column(item["metadata"]["type"], 'metadata.type') + "\t")
          + (parse_column(item["metadata"]["integrated"], 'metadata.integrated') + "\t")
          + (parse_column(item["metadata"]["member_databases"], 'metadata.member_databases') + "\t")
          + (parse_column(item["metadata"]["go_terms"], 'metadata.go_terms') + "\t")
          + (parse_column(item["proteins"][0]["accession"], 'proteins[0].accession') + "\t")
          + (parse_column(item["proteins"][0]["protein_length"], 'proteins[0].protein_length') + "\t")
          + (parse_column(item["proteins"][0]["entry_protein_locations"], 'proteins[0].entry_protein_locations')))
          row_l.append(row)
      if next:
          sleep(1)
    return row_l  

In [4]:
from tqdm.notebook import tqdm
acc_rows = []
for acc_num in tqdm(selected_prots, total=len(selected_prots)):
    if(acc_num is None):
        continue
    match_rows = output_list(acc_num)
    acc_rows.extend(match_rows)
acc_rows = [r.split('\t') for r in acc_rows]

  0%|          | 0/25 [00:00<?, ?it/s]

In [5]:
acc_rows[0]

['Q8NHC5',
 'G protein-coupled receptor, rhodopsin-like',
 'interpro',
 'family',
 '',
 'prosite:PS00237;smart:SM01381;pfam:PF00001;prints:PR00237',
 'GO:0004930,GO:0007186,GO:0016020',
 'q8nhc5',
 '309',
 '24..48,57..78,102..124,197..220,269..295']

In [6]:
domain_df = pd.DataFrame(acc_rows, columns=['PID', 'Name', 'Source DB', 'Type', 'Integrated', 'Member DB', 'GO Terms', 'Accession', 
                                     'Length', 'Protein Locations'])
domain_df = domain_df[domain_df['GO Terms'] != '']

import json
train_path = "/home/andrew/cafa5_team/data/"
with open(f"{train_path}/cafa_dataset/go_terms.json", "r") as f:
    go_terms = json.load(f)
domain_terms = [gt.split(',') for gt in domain_df['GO Terms']]
domain_term_ind = [[go_terms.index(t) for t in r if t in go_terms] for r in domain_terms]
domain_df['GO Ind'] = domain_term_ind
domain_df['Length'] = domain_df['Length'].map(int)
domain_df = domain_df[domain_df['Length'] < 800]

In [7]:
import numpy as np
go_domain = {}
for i, pid, go_terms, length, domain_loc in domain_df[['PID', 'GO Ind', 'Length', 'Protein Locations']].itertuples():
    for gi in go_terms:
        dom_id = (pid, gi)
        domain_locs = domain_loc.split(',')
        mask_update = np.zeros(length+1)
        for dl in domain_locs:
            si, ei = tuple([int(x) for x in dl.split('..')])
            mask_update[si:ei+1] = 1
        if(dom_id in go_domain):
            go_domain[dom_id] += mask_update
        else:
            go_domain[dom_id] = mask_update

In [8]:
go_domain_rows = [(pid, go_id, mask > 0) for (pid, go_id), mask in go_domain.items()]
go_domain_df = pd.DataFrame(go_domain_rows, columns=['Accession', 'GO ID', 'Domain Mask'])
mask_perc = np.array([m.mean() for m in go_domain_df['Domain Mask']])
go_domain_df = go_domain_df[(mask_perc <= 0.6) & (mask_perc >= 0.1)]

In [9]:
import requests
fasta_l = []
for acc_num in set(go_domain_df['Accession']):
    url = f"https://rest.uniprot.org/uniprotkb/{acc_num.upper()}.fasta"
    data = all_fastas = requests.get(url).text
    fasta_l.append(data)
with open("handpicked_interpro.fasta", "w") as f:
    f.writelines(fasta_l)
from go_bench.load_tools import load_protein_sequences
prot_seq, prot_id = load_protein_sequences('handpicked_interpro.fasta')
prot_dict = {pid:seq for pid, seq in zip(prot_id, prot_seq)}

In [10]:
with open('handpicked_dataset.pkl', 'wb') as f:
    pickle.dump((go_domain_df, prot_dict), f)

In [90]:
import numpy as np
import pandas as pd
from go_ml.train_utils import get_enzyme_df, enzyme_iterator, cls_seq_encode
import transformers
import matplotlib.pyplot as plt
tokenizer = transformers.AutoTokenizer.from_pretrained('facebook/esm2_t6_8M_UR50D')
from go_ml.go_utils import godag, go2parents_isa, get_ancestors

aa_str = 'LAGVSERTIDPKQNFYMHWC'
aa_ind = [tokenizer.get_vocab()[c] for c in aa_str]

import torch
device = torch.device('cuda:0')
from go_ml.models.bert_finetune import BERTFinetune
checkpoint_dir = "/home/andrew/GO_interp/checkpoints"
model = BERTFinetune.load_from_checkpoint(f"{checkpoint_dir}/esm_finetune-v1.ckpt", map_location=device)
model.eval()
print("Model loaded")

../go-basic.obo: fmt(1.2) rel(2024-04-24) 45,667 Terms


Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t33_650M_UR50D and are newly initialized: ['esm.pooler.dense.bias', 'esm.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded


In [93]:
from tqdm.notebook import tqdm
for prot_id, prot_seq in prot_dict.items():
    eval_ind = list(range(1, 1+len(prot_seq)))
    aa_str = 'LAGVSERTIDPKQNFYMHWC'
    aa_ind = [tokenizer.get_vocab()[c] for c in aa_str]
    eval_dict = {}
    seq_data = cls_seq_encode(prot_seq, tokenizer)
    with torch.no_grad():
        aa_ind = torch.tensor(aa_ind, device=device)
        seq_ind, mask =  torch.tensor(seq_data['seq_ind']).to(device), torch.BoolTensor(seq_data['mask']).to(device)
        base_batch_seq = torch.tile(seq_ind, (aa_ind.shape[0], 1))
        batch_mask = torch.tile(mask, (aa_ind.shape[0], 1))
        for res_ind in tqdm(eval_ind):
            batch_seq = base_batch_seq.clone()
            batch_seq[:, res_ind] = aa_ind
            logits = model.forward(batch_seq, batch_mask)
            eval_dict[res_ind] = logits.cpu().numpy()
    eval_mat = np.stack([eval_dict[i] for i in range(1, len(eval_dict)+1)])


  0%|          | 0/509 [00:00<?, ?it/s]

KeyboardInterrupt: 